# 03 — Cleanup and normalization
**Goal:** Remove extraction noise *without changing technical meaning*.

**Must:** repair obvious extraction artifacts, preserve originals.
**Trade-off:** aggressive normalization helps consistency but can corrupt identifiers, code, names, or exact-match keywords.

We deliberately preserve `$D SPOOL` and `ABEND=S0C4`.

In [ ]:
from pathlib import Path
import re
import unicodedata
DATA = Path('../data') if Path('../data').exists() else Path('data')
raw = (DATA / 'dirty_text.txt').read_text(encoding='utf-8')
print('RAW:')
print(repr(raw[:250]))

In [ ]:
def conservative_cleanup(text: str) -> str:
    text = unicodedata.normalize('NFC', text)
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    # Keep indentation and line boundaries; collapse repeated horizontal spaces only.
    text = re.sub(r'(?m)[ \t]{2,}', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    # Known repeating training footer; deliberately tailored, not a general regex.
    text = re.sub(r'(?m)^Page \d+ \| Internal Training\n?', '', text)
    return text.strip()

cleaned = conservative_cleanup(raw)
print(cleaned)
assert '$D SPOOL' in cleaned
assert 'ABEND=S0C4' in cleaned

### Exercise: inspect rather than blindly fix hyphenation
The split `utiliza-\ntion` might be a line-wrap artifact, while `long-\nterm` could have meaningful hyphenation. Write a *candidate detector* before an auto-repair rule.

In [ ]:
candidates = re.findall(r'([A-Za-z]+)-\n([A-Za-z]+)', cleaned)
print('Possible broken words:', candidates)
# Expected: [('utiliza', 'tion')]
# Production rule: flag for context-aware repair; do not blindly delete every hyphen.

**Exit check:** count removed boilerplate lines; compare original and cleaned versions; retain source offsets where practical.